# Versao 5 - Avaliacao Comparativa

## Finalidade

Este notebook executa a inferencia no conjunto de teste e consolida, em um mesmo quadro analitico, o desempenho dos tres metodos considerados:

- `LSTM pura`;
- `modelo hibrido residual`;
- `persistencia`.

A avaliacao e feita em regime `streaming`, o que preserva viabilidade computacional para um volume elevado de janelas temporais.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
PROJECT_ROOT = ROOT.parent if ROOT.name == "versao5" else ROOT
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
from versao5.pipeline_v5 import (
    evaluate_comparative_models,
    load_prepared_groups,
)

RUN_NAME = "comparativo_v5_artigo"
RUN_DIR = PROJECT_ROOT / "artifacts" / "reports_v5" / RUN_NAME
BUNDLE_PATH = RUN_DIR / "bundle_v5.json"
SPLIT_DIRECTORIES = {
    "train": RUN_DIR / "preprocessed" / "train",
    "validation": RUN_DIR / "preprocessed" / "validation",
    "test": RUN_DIR / "preprocessed" / "test",
}
MODEL_OUTPUT_DIR = RUN_DIR / "modelos"
EVAL_OUTPUT_DIR = RUN_DIR / "avaliacao"

bundle, groups = load_prepared_groups(BUNDLE_PATH, SPLIT_DIRECTORIES)
model_config_paths = sorted(MODEL_OUTPUT_DIR.glob("*/*_model_config.json"))
if not model_config_paths:
    raise FileNotFoundError(
        "Nenhum modelo treinado foi encontrado em MODEL_OUTPUT_DIR. "
        "Execute antes o notebook 3-modelagem-comparativa.ipynb."
    )
model_config_paths

## Avaliacao integrada

A funcao `evaluate_comparative_models` executa cada modelo no conjunto de teste e depois gera tabelas lado a lado para metricas globais, metricas por feature, metricas por classe, metricas por poco e metricas por horizonte.

In [ ]:
evaluations, comparison_tables = evaluate_comparative_models(
    bundle=bundle,
    test_groups=groups["test"],
    model_config_paths=model_config_paths,
    output_dir=EVAL_OUTPUT_DIR,
    batch_size=256,
    preview_rows=1024,
    progress_every=25,
    log_memory=False,
)

sorted(comparison_tables)

In [ ]:
display(comparison_tables["global_original"])
display(comparison_tables["per_feature_original"])

In [ ]:
display(comparison_tables["class_original"])
display(comparison_tables["horizon_original"])

## Orientacao para redacao do artigo

Recomenda-se organizar a secao de resultados da seguinte forma:

1. apresentar o quadro global (`global_original`);
2. discutir quais variaveis concentram os ganhos ou as perdas (`per_feature_original`);
3. examinar se o comportamento difere por classe de evento (`class_original`);
4. comentar a degradacao ou estabilidade ao longo do horizonte (`horizon_original`).

Esse encadeamento evita uma narrativa apenas agregada e produz uma avaliacao mais convincente do ponto de vista cientifico.

In [ ]:
ranking = (
    comparison_tables["global_original"]
    .sort_values("MAE")
    .reset_index(drop=True)
)
ranking

## Consideracoes finais

A interpretacao final deve distinguir entre:

- superioridade global agregada;
- robustez por feature;
- consistencia entre classes;
- custo computacional implicito na inferencia.

Um modelo pode melhorar o erro medio global e, ainda assim, falhar sistematicamente em certos regimes dinamicos. Por isso, o comparativo tridimensional entre `LSTM`, `hibrido residual` e `persistencia` constitui o nucleo empirico da versao5.